In [36]:
import pandas as pd
import numpy as np

df = pd.read_csv("./data/log_sensor.csv")
# df = pd.read_csv("./data/sintetis/dataset_mentah.csv")
print(f"Data: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head()

Data: 1777 baris, 9 kolom


,Timestamp,soil_moisture,soil_temperature,air_temperature,air_humidity,nitrogen,fosfor,kalium,ec
0,2026-07-04T01:32:00Z,82.5,25.3,24.8,58.0,91,190,102,1750
1,2026-07-04T01:37:00Z,80.5,25.2,24.8,57.1,93,187,101,1750
2,2026-07-04T01:42:00Z,77.9,25.6,25.3,53.5,91,190,101,1750
3,2026-07-04T01:47:00Z,78.2,25.8,25.0,55.4,94,188,103,1750
4,2026-07-04T01:52:00Z,75.1,26.0,26.0,51.5,93,189,100,1758


# Cek data

In [37]:
df[["soil_moisture", "soil_temperature", "air_temperature", "air_humidity"]].describe().round(2)

,soil_moisture,soil_temperature,air_temperature,air_humidity
count,1777.00,1777.00,1777.00,1777.00
mean,63.24,23.98,24.64,61.33
std,10.04,2.30,6.98,20.63
min,0.00,19.50,15.00,40.00
25%,57.70,22.00,17.50,40.00
50%,60.80,24.00,24.80,57.00
75%,65.30,25.90,31.70,83.10
max,95.00,28.30,34.00,99.00


In [38]:
# Konversi EC dari uS/cm -> mS/cm (dS/m) HANYA jika masih dalam ribuan
if df["ec"].max() > 100:        
    df["ec"] = (df["ec"] / 1000).round(2)

# Fungsi labelling

In [39]:
def label_irigasi(row):
    """
    Keputusan siram/tidak berbasis kelembaban tanah + konteks lingkungan.

    Referensi:
    - Threshold kelembaban 60-80% (Hatta 2006; skripsi Alvian Tabel 4.12)
    - Adaptif terhadap suhu & kelembaban udara (Rochmanto & Nursaputro, 2025)

    Logika:
    - Tanah kering (<60%) → siram
    - Tanah normal (60-80%) → default tidak, TAPI kalau panas + udara kering,
      siram lebih awal (antisipasi cepat kering)
    - Tanah basah (>80%) → jangan siram
    """
    sm = row["soil_moisture"]
    st = row["soil_temperature"]
    at = row["air_temperature"]
    ah = row["air_humidity"]

    if sm == 0:
        return -1

    # Threshold dasar: kering di bawah 60%
    threshold = 60

    # --- Adaptasi terhadap kondisi lingkungan ---
    # Suhu tanah tinggi → evaporasi cepat → naikkan threshold (siram lebih awal)
    
    if st > 30:
        threshold += 3

    # Suhu udara tinggi + kelembaban udara rendah → transpirasi tinggi
    if at > 32 and ah < 55:
        threshold += 4

    # Kelembaban udara sangat tinggi (lembab) → tunda, air bakal naik
    if ah > 85:
        threshold -= 5

    # Keputusan
    if sm >= 80:
        action = 0            # basah, jangan siram (cegah busuk akar)
    elif sm < threshold:
        action = 1            # kering (relatif kondisi) → siram
    else:
        action = 0            # normal → cukup

    return action

print("Fungsi label_irigasi() siap.")

Fungsi label_irigasi() siap.


# Terapkan labelling

In [40]:
df["irrigation_action"] = df.apply(label_irigasi, axis=1)

df = df[df["irrigation_action"] != -1].reset_index(drop=True) 

print("Distribusi label:")
dist = df["irrigation_action"].value_counts().sort_index()
for val, count in dist.items():
    label = "Tidak siram" if val == 0 else "Siram"
    print(f"  {val} ({label}): {count} ({count/len(df)*100:.1f}%)")


Distribusi label:
  0 (Tidak siram): 1094 (61.9%)
  1 (Siram): 674 (38.1%)


# Contoh

In [41]:
# Lihat baris yang keputusannya dipengaruhi faktor lingkungan (bukan cuma moisture)
sample = df[["soil_moisture", "soil_temperature", "air_temperature",
             "air_humidity", "irrigation_action"]].head(20)
sample

,soil_moisture,soil_temperature,air_temperature,air_humidity,irrigation_action
0,82.5,25.3,24.8,58.0,0
1,80.5,25.2,24.8,57.1,0
2,77.9,25.6,25.3,53.5,0
3,78.2,25.8,25.0,55.4,0
4,75.1,26.0,26.0,51.5,0
5,76.5,25.6,25.7,54.1,0
6,76.8,25.7,25.7,52.8,0
7,72.7,25.8,26.3,51.9,0
8,72.1,26.1,26.7,50.7,0
9,69.6,26.1,26.9,48.4,0


# Simpan

In [42]:
FEATURES = ["soil_moisture", "soil_temperature", "air_temperature", "air_humidity"]
cols = FEATURES + ["irrigation_action"]

df[cols].to_csv("data/dataset_irigasi.csv", index=False)
print("Tersimpan: data/dataset_irigasi.csv")
print(f"Total: {len(df)} baris, kolom: {cols}")


Tersimpan: data/dataset_irigasi.csv
Total: 1768 baris, kolom: ['soil_moisture', 'soil_temperature', 'air_temperature', 'air_humidity', 'irrigation_action']
